# Discovery and Success Criteria: Slow Down Before You Design

Riverside House asks for something that sounds simple: help editors find answers and continue manuscripts faster.

Your first job is not to choose a model. It is to understand the work well enough that Riverside can recognize a useful result and reject an unsafe one.

```mermaid
flowchart LR
    A["Riverside request"] --> B["Watch an editor work"]
    B --> C["Find delays and risky handoffs"]
    C --> D["Name decision owners"]
    D --> E["Write tests for success and failure"]
    E --> F["Decide whether design may begin"]
```

The notebook builds five in-memory discovery artifacts. Their internal keys help later chapters connect the work, but you do not need to memorize them. Focus on the questions they answer: who decides, how work happens today, what success means, what the system may do, and what is still unknown.

All people, organizations, metrics, and records are synthetic. Local fixture checks are not customer, cloud, security, legal, compliance, or production validation.

## 0 - The Riverside Challenge

Riverside has useful clues, but not enough agreement to design responsibly.

| What Riverside supplied | What is still missing |
|---|---|
| Policy lookup takes 18 minutes for the recorded sample | An accepted target and representative test cases |
| Continuation drafting takes 42 minutes for the recorded sample | A shared definition of an acceptable continuation |
| Current guidance ranked first in 61% of 36 replayed queries | Rules for rights questions, deletion, support, and workflow changes |

Those numbers are synthetic measured baselines. Keep their sample and limitations attached. They describe the fixture; they do not prove what Riverside should buy or what production will achieve.

```mermaid
flowchart LR
    A["Request"] --> B["Known today"]
    B --> C["Missing decisions"]
    C --> D["Small testable outcomes"]
    D --> E["Architecture input"]
```

**Predict before running code:** What hurts this workshop more: no preferred model, or no evidence about users, workflow, baseline, and authority?

In [ ]:
# ── Load and validate the frozen Riverside contract ──────────────────────
import json
from pathlib import Path
from jsonschema import validate

relative_chapter = Path("learning/role-based-tracks/fde/01-discovery-and-success-criteria")
chapter_dir = next((p.resolve() for p in [Path.cwd(), Path.cwd() / relative_chapter] if (p / "README.md").exists() and (p.parent / "shared").exists()), None)
if chapter_dir is None:
    raise FileNotFoundError("Run from the repository root or chapter directory.")
shared_dir = chapter_dir.parent / "shared"
case = json.loads((shared_dir / "fixtures/riverside-engagement-v1.json").read_text(encoding="utf-8"))
schema = json.loads((shared_dir / "schemas/riverside-engagement.schema.json").read_text(encoding="utf-8"))
facts = json.loads((shared_dir / "fixtures/expected-facts-v1.json").read_text(encoding="utf-8"))
validate(instance=case, schema=schema)
allowed_classes = {"measured_baseline", "modeled_assumption", "customer_claim", "policy_constraint", "external_validation_required", "intentional_conflict", "unknown"}
fde01_facts = [f for f in facts["facts"] if "FDE-01" in f["notebook_ids"]]
assert case["fixture_version"] == "RIV-FDE-1.0.0"
assert {f["evidence_class"] for f in fde01_facts} <= allowed_classes
print(f"PASS: schema-valid {case['fixture_version']} with {len(fde01_facts)} FDE-01 fact contracts")
print("  Local fixture validation is not customer or production validation.")

## 1 - See the Solution-First Failure

Imagine the opening workshop goes like this:

> Which model should we use? How many agents? Which vector database? What can we demo next week?

The team may leave with a polished backlog and still know nothing about the editor who needs help, the risky exceptions, or the person allowed to approve a change.

```mermaid
flowchart LR
    A["Model and demo choices"] --> B["Looks like progress"]
    B --> C["Workflow still unknown"]
    C --> D["No trustworthy success test"]
```

The next code cell makes that gap visible. Change one missing discovery dimension and notice that the other gaps remain. One good question helps; it does not replace discovery.

**Keep the labels honest:** a stakeholder statement is still a claim until it is checked. A calculation based on forecast traffic is still modeled, even when it appears in a dashboard.

In [ ]:
# ── Measure the solution-first coverage failure ──────────────────────────
solution_dimensions = {"solution", "demo"}
required_dimensions = {"users", "workflow", "baseline", "outcomes", "failure_cost", "authority", "data", "identity", "constraints", "acceptance", "ownership"}
missing = sorted(required_dimensions - solution_dimensions)
print(f"Discovery coverage: {len(required_dimensions & solution_dimensions)}/{len(required_dimensions)}")
print(f"Missing: {', '.join(missing)}")
print("Prediction confirmed: missing problem and authority evidence invalidates the workshop.")

new_dimension = "workflow"  # CHANGE THIS: choose one missing dimension
before = len(required_dimensions & solution_dimensions)
after = len(required_dimensions & (solution_dimensions | {new_dimension}))
assert new_dimension in required_dimensions and after == before + 1
print(f"Your turn PASS: coverage moves from {before} to {after}; the remaining gaps stay visible.")

## 2 - Put the Right People Around the Workflow

No single Riverside stakeholder can answer every discovery question.

```mermaid
flowchart TD
    E["Editor: where work breaks"] --> W["Editorial workflow"]
    D["Editorial Director: accepts workflow fit"] --> W
    R["Rights owner: decides permitted use"] --> W
    S["Security: protects imprint boundaries"] --> W
    I["IT: owns integration and support"] --> W
    F["Finance: accepts budget"] --> W
```

A sponsor can fund the work but may not know the daily exceptions. An editor can judge whether a suggestion helps but cannot approve manuscript access. The FDE can make the questions testable but cannot accept the workflow on Riverside's behalf.

**Predict:** Who should accept the editorial workflow: the sponsor, the Editorial Director, or the FDE? Choose by decision right, not job seniority.

The next cell builds the map from the synthetic case and checks that each important decision has an owner.

In [ ]:
# ── Build and check DSC-01 stakeholder and authority rows ────────────────
stakeholder_map = [{"person_id": p["person_id"], "persona": p["persona"], "role_ids": p["role_ids"], "tenant_ids": p["tenant_ids"], "goals": p["goals"], "concerns": p["concerns"], "decision_rights": p["decision_rights"], "evidence_class": "customer_claim", "source_ids": [p["person_id"], *p["statement_ids"]], "interview_status": "not_recorded_in_fixture"} for p in case["personas_and_stakeholders"]]
required_personas = {"executive sponsor", "business process owner", "editor champion", "legal and rights owner", "security approver", "integration and operations owner", "commercial approver"}
workflow_owner = next(row for row in stakeholder_map if "editorial workflow acceptance" in row["decision_rights"])
assert required_personas <= {row["persona"] for row in stakeholder_map}
assert workflow_owner["person_id"] == "PER-RIV-002"
assert all(row["decision_rights"] for row in stakeholder_map)
print(f"PASS: DSC-01 maps {len(stakeholder_map)} people with authority and tenant context")
print("Prediction resolved: the Editorial Director owns workflow acceptance.")

## 3 - Follow One Manuscript Through Today's Work

Riverside's work crosses email, shared files, rights records, chat, the editor, and PageTurn. The handoffs matter as much as the longest timer.

```mermaid
flowchart LR
    A["Receive 2m"] --> B["Search 18m"]
    B --> C["Verify 7m"]
    C --> D["Resolve 26m"]
    D --> E["Draft 42m"]
    E --> F["Review 15m"]
    F --> G["Update 4m"]
```

The obvious bottlenecks are search, resolving uncertainty, and drafting. But a shorter rights check may carry more risk than a long drafting step. Discovery looks at both time and consequence.

Do not add the seven median times and call the total an end-to-end median. The samples and timer boundaries differ, so that total would be a new unsupported claim.

The next cell keeps each baseline with its population, window, source, and limitation.

In [ ]:
# ── Build and check DSC-02 workflow and DSC-03 baseline ─────────────────
workflow = case["current_workflow"]
workflow_rows = [{**step, "evidence_class": workflow["evidence_class"], "source_ids": [workflow["workflow_id"], step["step_id"]]} for step in workflow["steps"]]
baseline_rows = [{**metric, "source_ids": [metric["metric_id"], metric["source_id"]]} for metric in case["baseline_metrics"]]
required = {"value", "unit", "population", "window", "source_id", "evidence_class", "limitations"}
naive_sum = sum(row["median_minutes"] for row in workflow_rows)
assert [row["sequence"] for row in workflow_rows] == list(range(1, 8))
assert all(row["failure_modes"] for row in workflow_rows)
assert all(required <= row.keys() and row["evidence_class"] == "measured_baseline" for row in baseline_rows)
print(f"PASS: DSC-02 has {len(workflow_rows)} steps; DSC-03 has {len(baseline_rows)} bounded metrics")
print(f"  Diagnostic only: step medians sum to {naive_sum:.0f} minutes; do not report this as end-to-end median.")

## 4 - Draw the Boundary Before Connecting Data

Riverside wants a better workflow, not a particular feature. Start with the result, then make each action boundary visible.

```mermaid
flowchart LR
    O["Editor needs help"] --> R["Assistant retrieves or drafts"]
    R --> P["Assistant proposes exact change"]
    P --> H["Human reviews and approves"]
    H --> C["Authorized service commits once"]
    C --> A["Audit and correct if needed"]
```

This prevents a vague sentence such as "the assistant updates PageTurn" from hiding who approved the payload, which identity performed the write, and how a retry avoids a duplicate change.

The same caution applies to data. A source appearing in an inventory means Riverside knows it exists. It does not mean the source is approved, current, correctly permissioned, available in the right region, or able to honor deletion.

**Predict:** If a source has an owner and sample record IDs, is it approved for this use? The next cell checks the difference between visibility and permission.

In [ ]:
# ── Build and check scope, action, data, and constraint inventories ──────
brief = case["ambiguous_brief"]
outcome_rows = [{"outcome_id": f"OUT-RIV-{i:03d}", "statement": s, "evidence_class": brief["evidence_class"], "source_ids": [brief["brief_id"]], "status": "draft_requires_validation"} for i, s in enumerate(brief["explicit_outcomes"], 1)]
non_goal_rows = [{**item, "evidence_class": "policy_constraint", "source_ids": [item["non_goal_id"]]} for item in case["non_goals"]]
bounds = {"UC-RIV-001": ("retrieve_with_citations", "ROLE-EDITOR", "none"), "UC-RIV-002": ("draft_only", "ROLE-EDITOR", "none"), "UC-RIV-003": ("propose_exact_payload", "ROLE-EDITOR or ROLE-EDITORIAL-DIRECTOR", "SYS-WORKFLOW after human confirmation"), "UC-RIV-004": ("retrieve_restrictions_only", "ROLE-RIGHTS-COUNSEL", "none")}
action_inventory = []
for uc in case["use_cases"]:
    assistant, approval, commit = bounds[uc["use_case_id"]]
    action_inventory.append({"action_id": uc["use_case_id"].replace("UC-", "ACT-"), "use_case_id": uc["use_case_id"], "actor_role_ids": uc["actor_role_ids"], "assistant_role": assistant, "approval_authority": approval, "commit_boundary": commit, "human_action": uc["human_action"], "decision_boundary": uc["decision_boundary"], "required_context": case["identity_and_data_constraints"]["required_request_context"], "source_ids": [uc["use_case_id"]], "status": "draft"})
constraint_rows = [{**item, "source_ids": [item["constraint_id"]]} for item in case["identity_and_data_constraints"]["global_constraints"]]
systems = {item["system_id"]: item for item in case["systems"]}
data_inventory = [{"source_id": src["source_id"], "system_id": src["system_id"], "source_owner_person_id": src["owner_person_id"], "system_owner_person_id": systems[src["system_id"]]["owner_person_id"], "source_type": src["source_type"], "classification": src["classification"], "tenant_ids": src["tenant_ids"], "region_ids": src["region_ids"], "acl_model": src["acl_model"], "update_mode": src["update_mode"], "deletion_behavior": src["deletion_behavior"], "seeded_failures": src["seeded_failures"], "evidence_class": src["evidence_class"], "approval_status": "inventory_only_not_approved", "source_ids": [src["source_id"], src["system_id"]]} for src in case["source_inventory"]]
assert (len(outcome_rows), len(non_goal_rows), len(action_inventory), len(data_inventory)) == (2, 4, 4, 6)
assert "human confirmation" in next(row for row in action_inventory if row["use_case_id"] == "UC-RIV-003")["commit_boundary"]
assert all(row["evidence_class"] == "customer_claim" and row["approval_status"].endswith("not_approved") for row in data_inventory)
assert any(row["evidence_class"] == "external_validation_required" for row in constraint_rows)
print("PASS: outcomes stay technology-neutral; non-goals, actions, sources, and constraints stay bounded")
print("Prediction resolved: zero sources are approved by inventory alone.")
selected_source_id = "SRC-API-WORKFLOW-001"  # CHANGE THIS: choose another SRC-* ID
selected = next(row for row in data_inventory if row["source_id"] == selected_source_id)
print(f"Your turn: {selected_source_id} has {len(selected['seeded_failures'])} failure hypotheses to test.")

## 5 - Test the Cases Riverside Cannot Average Away

A useful success criterion answers five plain questions:

1. What should improve or never happen?
2. Which editors, imprints, regions, and edge cases are included?
3. How will Riverside test it?
4. Who decides whether the result is acceptable?
5. Is the target measured, modeled, customer-validated, or still unknown?

```mermaid
flowchart LR
    A["Outcome"] --> B["Representative cases"]
    B --> C["Test method"]
    C --> D{"Any forbidden failure?"}
    D -->|Yes| E["Block or narrow release"]
    D -->|No| F["Authorized review"]
```

Suppose 99 authorization checks pass and one editor retrieves another imprint's embargoed manuscript. The average is 99%, but the release is blocked. A must-pass safety boundary is not an average quality target.

**Predict:** Will 99 out of 100 pass? Run the next cell and compare the aggregate score with the forbidden-failure count.

In [ ]:
# ── Prove the slice failure and draft DSC-04 criteria ───────────────────
authorization = [True] * 99 + [False]
aggregate = sum(authorization) / len(authorization)
forbidden = authorization.count(False)
assert aggregate >= 0.99 and forbidden != 0
print(f"Aggregate {aggregate:.0%} passes; forbidden access {forbidden} fails. Release BLOCKED.")
acceptance_criteria = [
 {"criterion_id": "AC-RIV-001", "outcome": "current policy first", "metric": "current_guidance_first_result_rate", "baseline_ref": "MET-RIV-005", "target": None, "target_class": "unknown", "slices": ["tenant", "imprint", "policy lifecycle"], "method": "versioned replay", "owner_person_id": "PER-RIV-002", "source_ids": ["MET-RIV-005", "UNK-RIV-001", "UNK-RIV-007"], "status": "blocked_pending_target"},
 {"criterion_id": "AC-RIV-002", "outcome": "authorized cited answers", "metric": "citation_support_and_authorization_rate", "baseline_ref": None, "target": None, "target_class": "unknown", "slices": ["policy", "rights", "no-evidence abstention"], "method": "labeled citation review", "owner_person_id": "PER-RIV-002/PER-RIV-003", "source_ids": ["UC-RIV-001", "UC-RIV-004", "CON-RIV-003"], "status": "blocked_pending_rubric"},
 {"criterion_id": "AC-RIV-003", "outcome": "faster continuation without extra rework", "metric": "draft_minutes_and_rework", "baseline_ref": "MET-RIV-003/MET-RIV-004", "target": None, "target_class": "unknown", "slices": ["length", "genre", "editor experience"], "method": "paired timing and rejection reasons", "owner_person_id": "PER-RIV-002", "source_ids": ["MET-RIV-003", "MET-RIV-004", "UNK-RIV-007"], "status": "blocked_pending_quality_floor"},
 {"criterion_id": "AC-RIV-004", "outcome": "no forbidden retrieval", "metric": "successful_forbidden_accesses", "baseline_ref": None, "target": 0, "target_class": "policy_constraint", "slices": ["tenant", "role", "region", "purpose", "title", "contractor"], "method": "negative isolation tests", "owner_person_id": "PER-RIV-004", "source_ids": ["SLA-RIV-005", "SEC-RIV-002", "RISK-RIV-003"], "status": "draft_must_pass"},
 {"criterion_id": "AC-RIV-005", "outcome": "one approved transition commits once", "metric": "duplicate_workflow_commits", "baseline_ref": "MET-RIV-006", "target": 0, "target_class": "policy_constraint", "slices": ["timeout before/after commit", "retry", "reconciliation"], "method": "failure injection", "owner_person_id": "PER-RIV-005", "source_ids": ["SLA-RIV-006", "SEC-RIV-004", "RISK-RIV-005"], "status": "draft_must_pass_before_writes"},
 {"criterion_id": "AC-RIV-006", "outcome": "lookup fits proposed envelope", "metric": "policy_lookup_latency_p95", "baseline_ref": None, "target": 8.0, "target_class": "modeled_assumption", "slices": ["UK/EU", "US", "cache", "context size"], "method": "representative load test", "owner_person_id": "PER-RIV-005", "source_ids": ["SLA-RIV-001", "CON-RIV-001", "UNK-RIV-008"], "status": "proposed_requires_measurement"}
]
required = {"criterion_id", "metric", "target_class", "slices", "method", "owner_person_id", "source_ids", "status"}
assert all(required <= row.keys() and row["slices"] for row in acceptance_criteria)
assert all(row["target_class"] != "customer_validated" for row in acceptance_criteria)
golden_workflow_drafts = [
 {"case_id": "GW-RIV-001", "slice": "EU/current policy", "expected": "authorized current citation", "source_ids": ["UC-RIV-001", "TEN-RIV-EU"]},
 {"case_id": "GW-RIV-002", "slice": "US/UK-only policy", "expected": "exclude inapplicable guidance", "source_ids": ["CON-RIV-011", "TEN-RIV-US"]},
 {"case_id": "GW-RIV-003", "slice": "rights/no evidence", "expected": "abstain and escalate", "source_ids": ["UC-RIV-004", "PER-RIV-003"]},
 {"case_id": "GW-RIV-004", "slice": "assigned continuation", "expected": "draft without save/publish", "source_ids": ["UC-RIV-002", "NG-RIV-001"]},
 {"case_id": "GW-RIV-005", "slice": "cross-tenant request", "expected": "deny and audit", "source_ids": ["SEC-RIV-002", "RISK-RIV-003"]},
 {"case_id": "GW-RIV-006", "slice": "timeout after commit", "expected": "reconcile before retry", "source_ids": ["SEC-RIV-004", "RISK-RIV-005"]},
 {"case_id": "GW-RIV-007", "slice": "withdrawn draft", "expected": "exclude within unknown bound", "source_ids": ["CON-RIV-006", "UNK-RIV-002"]},
 {"case_id": "GW-RIV-008", "slice": "regional outage", "expected": "fail closed", "source_ids": ["SEC-RIV-001", "UNK-RIV-003"]}
]
for row in golden_workflow_drafts:
    row.update(evidence_class="policy_constraint", status="draft_requires_validation")
print(f"PASS: {len(acceptance_criteria)} criteria and {len(golden_workflow_drafts)} golden workflow drafts remain honestly provisional.")

## 6 - Turn Uncertainty Into the Next Conversation

Riverside's open questions should drive discovery, not disappear into a long appendix.

| If the answer could... | Investigate it... | Riverside example |
|---|---|---|
| Expose forbidden data | First | How quickly must withdrawn manuscripts disappear from every index? |
| Change the architecture | Early | Can PageTurn prevent duplicate workflow updates after a timeout? |
| Refine a later choice | After blockers | Which dashboard layout do editors prefer? |

Keep both sides of a disagreement until the authorized owner decides. Code must not settle a dispute about rights, deletion, quality, or support just because one answer is easier to implement.

```mermaid
flowchart LR
    U["Open question"] --> I["Evidence needed"]
    I --> O["Named owner"]
    O --> N["Needed before a decision"]
```

The next cell groups the fixture's assumptions, conflicts, risks, and unknowns, then prioritizes the questions most likely to block safety or change the design.

In [ ]:
# ── Build and prioritize DSC-05 without closing uncertainty ─────────────
assumptions = [{**x, "source_ids": [x["assumption_id"]], "status": "open_modeled_input"} for x in case["demand_cost_and_sla_assumptions"]["assumptions"]]
conflicts = [{**x, "source_ids": [x["conflict_id"]], "status": "open_requires_decision_artifact"} for x in case["intentional_conflicts"]]
risks = [{**x, "evidence_class": "policy_constraint", "source_ids": [x["risk_id"]], "status": "open"} for x in case["seeded_risks"]]
unknowns = [{**x, "evidence_class": "external_validation_required" if x["status"] == "external_validation_required" else "unknown", "source_ids": [x["unknown_id"]]} for x in case["unknowns"]]
critical = ("deletion", "entitlement", "coverage", "quality", "golden workflow")
architecture = ("idempotency", "fail over", "hosted model", "metadata")
def priority(question):
    text = question.lower()
    return 1 if any(term in text for term in critical) else 2 if any(term in text for term in architecture) else 3
backlog = sorted(({"backlog_id": x["unknown_id"], "question": x["question"], "priority": priority(x["question"]), "owner_person_id": x["owner_person_id"], "needed_by": x["needed_by"], "evidence_class": x["evidence_class"], "status": x["status"], "source_ids": x["source_ids"]} for x in unknowns), key=lambda x: (x["priority"], x["needed_by"], x["backlog_id"]))
assert (len(assumptions), len(conflicts), len(risks), len(unknowns)) == (9, 11, 8, 10)
assert all(x["owner_person_id"] and x["needed_by"] for x in backlog)
print("PASS: DSC-05 retains 9 assumptions, 11 conflicts, 8 risks, and 10 unknowns.")
backlog_id = "UNK-RIV-005"  # CHANGE THIS: choose UNK-RIV-001 through UNK-RIV-010
item = next(x for x in backlog if x["backlog_id"] == backlog_id)
print(f"Your turn: {item['question']} -> {item['owner_person_id']} before {item['needed_by']}")

## 7 - Decide Whether Riverside Is Ready to Design

A complete worksheet can still end with the right answer: stop.

Riverside now has an organized view of users, workflow, evidence, actions, candidate data, and tests. It still lacks accepted quality targets, representative cases, workflow-write scope, deletion timing, support coverage, and some external capacity evidence.

```mermaid
flowchart LR
    A["People and authority"] --> R["Discovery review"]
    B["Workflow and baseline"] --> R
    C["Success and safety tests"] --> R
    D["Owned open questions"] --> R
    R -->|Critical decisions open| X["BLOCKED"]
    R -->|Owners decide and evidence is ready| Y["Compare architectures"]
```

Use [templates/discovery-review.md](templates/discovery-review.md) for the conversation. A person attending a review is not the same as that person approving a decision. Record the authorized role, reviewed scope, decision, conditions, date, and reason to revisit it.

In [ ]:
# ── Assemble the discovery pack and evaluate the gate honestly ──────────
discovery_pack = {
 "engagement_id": case["engagement"]["engagement_id"], "fixture_version": case["fixture_version"],
 "DSC-01": {"stakeholder_map": stakeholder_map},
 "DSC-02": {"workflow_id": workflow["workflow_id"], "steps": workflow_rows},
 "DSC-03": {"baseline_metrics": baseline_rows},
 "DSC-04": {"outcomes": outcome_rows, "non_goals": non_goal_rows, "actions": action_inventory, "constraints": constraint_rows, "criteria": acceptance_criteria, "golden_workflows": golden_workflow_drafts},
 "DSC-05": {"assumptions": assumptions, "conflicts": conflicts, "risks": risks, "unknowns": unknowns, "backlog": backlog, "data_inventory": data_inventory}
}
blocking = [x["criterion_id"] for x in acceptance_criteria if x["status"].startswith("blocked")]
gate = {"status": "BLOCKED" if blocking or conflicts else "READY_FOR_REVIEW", "blocking_criteria": blocking, "open_conflicts": [x["conflict_id"] for x in conflicts], "open_unknowns": [x["unknown_id"] for x in unknowns]}
assert set(discovery_pack) >= {"DSC-01", "DSC-02", "DSC-03", "DSC-04", "DSC-05"}
assert gate["status"] == "BLOCKED"
print("PASS: DSC-01 through DSC-05 assembled with frozen provenance.")
print(f"Discovery gate: {gate['status']} - {len(blocking)} blocking criteria, {len(conflicts)} conflicts, {len(unknowns)} unknowns")
print("  Useful artifacts do not manufacture permission to design or deploy.")

## 8 - What Changed for Riverside

```mermaid
flowchart LR
    A["Build an AI system"] --> B["Help named editors in a known workflow"]
    B --> C["Retrieve and draft within clear boundaries"]
    C --> D["Test normal, difficult, and forbidden cases"]
    D --> E["Resolve the decisions that still block design"]
```

### Riverside takeaways

1. Start with an editor's current decision and workflow, not the requested technology.
2. Keep measured results, modeled forecasts, and customer decisions separate.
3. Ask frontline users and decision owners different questions.
4. Keep every baseline attached to its sample, method, and limitation.
5. Separate suggestion, approval, commit, audit, and correction.
6. Never let a good average hide one forbidden access or action.
7. Treat `BLOCKED` as useful evidence when important decisions are still open.

The next chapter compares process changes, deterministic software, search, retrieval with generation, fine-tuning, workflows, and agentic designs. Riverside's discovery boundaries will rule out options that cannot preserve access, approval, and operational ownership.

For an authorized run, use [templates/notebook-output-record.md](templates/notebook-output-record.md). Leave it as `NOT RUN` until results are actually observed.